# Fine-Tuning: multilingual-e5-large on TripLegal-CL


**Objective.** The central claim of the paper is that TripLegal-CL provides
an effective contrastive training signal for adapting dense bi-encoder
retrievers to the Spanish legal domain. To validate this, we fine-tune
`intfloat/multilingual-e5-large` — a strong general-purpose multilingual
encoder — using contrastive learning on TripLegal-CL, and then evaluate
it on the **same benchmark** used for its baseline. If the fine-tuned
model consistently outperforms the baseline across all IR metrics, this
confirms that the corpus provides useful domain-specific supervision.


**Dev evaluator.** During training, a dev evaluator (50K queries, 80K
corpus) runs every 100 steps to monitor convergence. It is drawn from
the **training region** (first 380K instances) and is intentionally
smaller for speed.

# 1. Environment Setup

In [1]:
!pip install -U "sentence-transformers>=3.0.1" "transformers>=4.48.0" datasets accelerate

## 2. Imports

In [2]:
import logging
import traceback

from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.losses import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)

## 3. Load Model

**Note on E5 prefixes:** multilingual-e5-large requires "query: " and
"passage: " prefixes. In the original notebook, these were manually
added in the `map()` function. Here, we handle them via `prompts={}` in
the training arguments (Step 6), which is cleaner and less error-prone.


In [3]:
model = SentenceTransformer(
    "intfloat/multilingual-e5-large",
    model_card_data=SentenceTransformerModelCardData(
        language="es",
        license="apache-2.0",
        model_name="multilingual-e5-large trained on TripLegal-CL Legal Spanish.",
    ),
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4. Prepare Data Helper

**Key difference from original:** No "query: " or "passage: " prefixes
here. The raw text is stored; prefixes are applied automatically by the
trainer via the `prompts` parameter.

In [4]:
def select_first_pos(example):
    if example["pos"]:
        return {"query": example["query"], "pos": example["pos"][0]}

## 5. Load and Split Dataset

In [5]:
SEED = 42

TRAIN_N = 300_000
EVAL_N  = 50_000
TEST_N  = 30_000

base = load_dataset("wilfredomartel/TripLegal-CL", split="train").shuffle(seed=SEED)

# Disjoint ranges — no data leakage
train_dataset = base.select(range(0, TRAIN_N))
eval_dataset  = base.select(range(TRAIN_N, TRAIN_N + EVAL_N))
test_dataset  = base.select(range(TRAIN_N + EVAL_N, TRAIN_N + EVAL_N + TEST_N))

cols_to_remove = ["neg", "pos_score", "neg_score"]

train_dataset = train_dataset.remove_columns(cols_to_remove).map(select_first_pos)
eval_dataset  = eval_dataset.remove_columns(cols_to_remove).map(select_first_pos)
test_dataset  = test_dataset.remove_columns(cols_to_remove).map(select_first_pos)

print(f"Train: {len(train_dataset):,}")
print(f"Eval:  {len(eval_dataset):,}")
print(f"Test:  {len(test_dataset):,}")
print(train_dataset[0])

Map:   0%|          | 0/300000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Train: 300,000
Eval:  50,000
Test:  30,000
{'query': '¿Por qué la Primera Sala de la Suprema Corte de Justicia de la Nación declaró infundado el recurso de reclamación 456/2019, confirmando el desechamiento del recurso de revisión?', 'pos': 'La Primera Sala de la Suprema Corte de Justicia de la Nación declaró infundado el recurso de reclamación 456/2019, confirmando el desechamiento del recurso de revisión, al determinar que los agravios presentados por Manuel Contreras Ramos no combatían las razones del acuerdo de presidencia recurrido. El acuerdo de desechamiento se basó en la inexistencia de una cuestión propiamente constitucional, mientras que los agravios del recurrente se enfocaron en demostrar la importancia y trascendencia del asunto, sin desvirtuar la falta de un tema de constitucionalidad. La Sala aplicó la tesis 1a. XXXVI/2018 (10a.) para señalar que los agravios que no desvirtúan la inexistencia de una cuestión constitucional son inoperantes, y que la falta de un tema de co

## 6. Loss Function and Training Arguments

In [6]:
# CachedMNRL: same loss as Gemma notebook
# mini_batch_size controls GPU memory; per_device_train_batch_size controls
# the effective number of in-batch negatives
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16)

run_name = "multilingual-e5-large-es-legal-300k-v3"

args = SentenceTransformerTrainingArguments(
    output_dir=f"models/{run_name}",
    num_train_epochs=1,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    warmup_ratio=0.03,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    # E5 prefixes — handled here, NOT in the data
    prompts={
        "query": "query: ",
        "pos": "passage: ",
    },
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=20,
    run_name=run_name,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


### How contrastive learning works in this training setup

This section explains how the contrastive training signal is formed
during fine-tuning — the mechanism referred to as "contrastive learning"
in the paper.

**Training data format.** Each training example is a `(query, pos)` pair.
We do NOT explicitly provide negative passages to the trainer. Instead,
negatives are constructed automatically at training time through a
mechanism called **in-batch negatives**.

**In-batch negatives (InfoNCE / MNRL).** Given a batch of `B` pairs
`{(q₁, p₁), (q₂, p₂), ..., (qB, pB)}`, the loss function:

1. Encodes all `B` queries and all `B` passages into dense vectors.
2. Computes a `B × B` cosine similarity matrix between all queries
   and all passages.
3. For each query `qᵢ`, the passage `pᵢ` is the **positive** (correct
   answer), and all other passages `{p₁, ..., pᵢ₋₁, pᵢ₊₁, ..., pB}`
   are treated as **negatives** (wrong answers).
4. Applies cross-entropy loss to push `qᵢ` closer to `pᵢ` and away
   from all other passages in the batch.

```
Similarity matrix (batch_size=4 example):

              p₁     p₂     p₃     p₄
        q₁ [ 0.92   0.45   0.51   0.38 ]  ← maximize (q₁, p₁)
        q₂ [ 0.41   0.89   0.47   0.52 ]  ← maximize (q₂, p₂)
        q₃ [ 0.50   0.43   0.91   0.40 ]  ← maximize (q₃, p₃)
        q₄ [ 0.39   0.48   0.42   0.87 ]  ← maximize (q₄, p₄)

Diagonal = positive pairs (should be highest in each row)
Off-diagonal = in-batch negatives (should be lower)



This means that with a batch size of 128, each query has **127 implicit
negatives** per training step — all from the same legal domain, making
them naturally hard negatives.

**Why larger batches improve performance.** More samples per batch =
more in-batch negatives = harder contrastive signal = better
discrimination. This is why `CachedMultipleNegativesRankingLoss` is
valuable: it allows an effective batch size of 128 while only using
the GPU memory of `mini_batch_size=16`, by caching intermediate
embeddings (GradCache; Gao et al., 2021).

**Role of `BatchSamplers.NO_DUPLICATES`.** This sampler ensures that
no two samples in the same batch share identical text (query or
passage). This is critical because:

- If `p₃ == p₇` (duplicate passages in the batch), then `q₃` would
  have its own correct answer appearing as a "negative" — sending a
  contradictory gradient signal to the model.
- `NO_DUPLICATES` prevents this by checking for text duplicates when
  forming each batch.

**Important:** `NO_DUPLICATES` is a **sampler** (controls batch
composition), not a loss function. It does not generate negatives —
the loss function does that via the similarity matrix above.

Summary of roles:

| Component | Role |
|-----------|------|
| `CachedMNRL` (loss) | Constructs in-batch negatives from the B×B similarity matrix and computes cross-entropy |
| `NO_DUPLICATES` (sampler) | Ensures no duplicate texts in a batch, preventing false negatives |
| `per_device_train_batch_size=128` | Controls how many in-batch negatives each query sees (127) |
| `mini_batch_size=16` | Controls GPU memory usage (forward pass in chunks of 16) |


## 7. Build Dev Evaluator (monitoring during training)

This evaluator runs every 100 training steps to monitor convergence.
It uses data from the **training region** (first 380K instances), NOT
from the final evaluation region. It is intentionally smaller (50K
queries, 80K corpus) for speed.

In [7]:
queries = dict(enumerate(eval_dataset["query"]))

corpus_list = eval_dataset["pos"][:] + train_dataset.select(range(30_000))["pos"][:]
corpus = dict(enumerate(corpus_list))

relevant_docs = {idx: [idx] for idx in queries}

dev_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legal-spanish-e5-eval-50kq-80kd",
    show_progress_bar=True,
)

# Evaluate base model before training
dev_evaluator(model)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  50%|█████     | 1/2 [08:18<08:18, 498.23s/it]

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 2/2 [13:18<00:00, 399.42s/it]


{'legal-spanish-e5-eval-50kq-80kd_cosine_accuracy@1': 0.79096,
 'legal-spanish-e5-eval-50kq-80kd_cosine_accuracy@3': 0.8672,
 'legal-spanish-e5-eval-50kq-80kd_cosine_accuracy@5': 0.89008,
 'legal-spanish-e5-eval-50kq-80kd_cosine_accuracy@10': 0.9142,
 'legal-spanish-e5-eval-50kq-80kd_cosine_precision@1': 0.79096,
 'legal-spanish-e5-eval-50kq-80kd_cosine_precision@3': 0.2890666666666666,
 'legal-spanish-e5-eval-50kq-80kd_cosine_precision@5': 0.17801600000000004,
 'legal-spanish-e5-eval-50kq-80kd_cosine_precision@10': 0.09142000000000002,
 'legal-spanish-e5-eval-50kq-80kd_cosine_recall@1': 0.79096,
 'legal-spanish-e5-eval-50kq-80kd_cosine_recall@3': 0.8672,
 'legal-spanish-e5-eval-50kq-80kd_cosine_recall@5': 0.89008,
 'legal-spanish-e5-eval-50kq-80kd_cosine_recall@10': 0.9142,
 'legal-spanish-e5-eval-50kq-80kd_cosine_ndcg@10': 0.8535012078412976,
 'legal-spanish-e5-eval-50kq-80kd_cosine_mrr@10': 0.8339754285714233,
 'legal-spanish-e5-eval-50kq-80kd_cosine_map@100': 0.8363526384595287}

## 8. Train

In [8]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    evaluator=dev_evaluator,
)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Legal-spanish-e5-eval-50kq-80kd Cosine Accuracy@1,Legal-spanish-e5-eval-50kq-80kd Cosine Accuracy@3,Legal-spanish-e5-eval-50kq-80kd Cosine Accuracy@5,Legal-spanish-e5-eval-50kq-80kd Cosine Accuracy@10,Legal-spanish-e5-eval-50kq-80kd Cosine Precision@1,Legal-spanish-e5-eval-50kq-80kd Cosine Precision@3,Legal-spanish-e5-eval-50kq-80kd Cosine Precision@5,Legal-spanish-e5-eval-50kq-80kd Cosine Precision@10,Legal-spanish-e5-eval-50kq-80kd Cosine Recall@1,Legal-spanish-e5-eval-50kq-80kd Cosine Recall@3,Legal-spanish-e5-eval-50kq-80kd Cosine Recall@5,Legal-spanish-e5-eval-50kq-80kd Cosine Recall@10,Legal-spanish-e5-eval-50kq-80kd Cosine Ndcg@10,Legal-spanish-e5-eval-50kq-80kd Cosine Mrr@10,Legal-spanish-e5-eval-50kq-80kd Cosine Map@100
100,0.023593,0.016337,0.903620,0.949520,0.961560,0.973960,0.903620,0.316507,0.192312,0.097396,0.903620,0.949520,0.961560,0.973960,0.940023,0.929031,0.929998
200,0.012973,0.011957,0.912660,0.956000,0.967960,0.978920,0.912660,0.318667,0.193592,0.097892,0.912660,0.956000,0.967960,0.978920,0.946958,0.936591,0.937462
300,0.011727,0.011658,0.916060,0.957240,0.967580,0.979440,0.916060,0.319080,0.193516,0.097944,0.916060,0.957240,0.967580,0.979440,0.948692,0.938755,0.939592
400,0.020286,0.010737,0.919640,0.960020,0.970420,0.980760,0.919640,0.320007,0.194084,0.098076,0.919640,0.960020,0.970420,0.980760,0.951445,0.941924,0.942751
500,0.014170,0.009497,0.920500,0.959980,0.970920,0.981680,0.920500,0.319993,0.194184,0.098168,0.920500,0.959980,0.970920,0.981680,0.952018,0.942423,0.943178
600,0.013191,0.008885,0.923800,0.962380,0.972680,0.982860,0.923800,0.320793,0.194536,0.098286,0.923800,0.962380,0.972680,0.982860,0.954370,0.945137,0.945861
700,0.011404,0.008554,0.924900,0.962700,0.972600,0.982380,0.924900,0.320900,0.194520,0.098238,0.924900,0.962700,0.972600,0.982380,0.954746,0.945780,0.946565
800,0.005816,0.007672,0.927600,0.965540,0.975180,0.984640,0.927600,0.321847,0.195036,0.098464,0.927600,0.965540,0.975180,0.984640,0.957285,0.948396,0.949057
900,0.008466,0.007483,0.926340,0.964960,0.975260,0.984980,0.926340,0.321653,0.195052,0.098498,0.926340,0.964960,0.975260,0.984980,0.956782,0.947626,0.948277
1000,0.010534,0.006949,0.930640,0.967660,0.976900,0.986340,0.930640,0.322553,0.195380,0.098634,0.930640,0.967660,0.976900,0.986340,0.959723,0.951066,0.951687


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 2/2 [02:06<00:00, 63.28s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2344, training_loss=0.03450764444578795, metrics={'train_runtime': 15663.0167, 'train_samples_per_second': 19.153, 'train_steps_per_second': 0.15, 'total_flos': 0.0, 'train_loss': 0.03450764444578795, 'epoch': 1.0})

## 9. Post-Training Evaluation and Save

Running the evaluators after training serves two purposes: (1) confirm
final metrics, and (2) **log results into the model card** — Hugging Face
automatically includes the last evaluation scores in the model card
when pushing to the Hub.

We evaluate on both the **dev set** (50K queries) and a held-out **test
set** (30K queries, 50K corpus) to provide two independent performance
snapshots in the model card.

In [9]:
# Re-run dev evaluator — results are logged into the model card
dev_evaluator(model)

queries = dict(enumerate(test_dataset["query"]))
corpus_list = test_dataset["pos"][:] + train_dataset.select(range(20_000))["pos"][:]
corpus = dict(enumerate(corpus_list))

relevant_docs = {idx: [idx] for idx in queries}
test_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="legal-spanish-eval-30kq-50kd",
    show_progress_bar=True,
)
test_evaluator(model)

# Save the trained model
final_output_dir = f"models/{run_name}/final"
model.save_pretrained(final_output_dir)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks:  50%|█████     | 1/2 [01:18<01:18, 78.06s/it]

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/938 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [01:17<00:00, 77.00s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## 10. Push to Hub

In [10]:
try:
    model.push_to_hub(run_name)
except Exception:
    logging.error(
        f"Error uploading model to Hub:\n{traceback.format_exc()}"
        f"Model is saved locally at: {final_output_dir}\n"
        f"To retry: model = SentenceTransformer('{final_output_dir}')\n"
        f"Then: model.push_to_hub('{run_name}')"
    )

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpjwfv2_zr/tokenizer.json: 100%|##########| 16.8MB / 16.8MB            

  ...wfv2_zr/model.safetensors:   0%|          |  104kB / 2.24GB            